<style>
.jp-RenderedHTMLCommon img, .rendered_html img {
  display: block;
  margin-left: 0 !important;
  margin-right: auto !important;
}
</style>

# Image analysis with napari

_Written by [Stella Prins](https://profiles.ucl.ac.uk/54400-stella-prins)_

In this tutorial, I will revisit three examples from the image analysis workshop that [Prof Dave Cash](https://profiles.ucl.ac.uk/15006-dave-cash) and I delivered at the AI Catalyst 2026 conference in York, using [napari](https://napari.org/), an open-source tool for visualising and analysing images.

In **Example 1**, we will segment fluorescent nuclei using traditional image-processing methods. In **Example 2**, we will see what happens when we apply the same approach to unlabelled live cells. Finally, in **Example 3**, we will explore how [micro-SAM](https://computational-cell-analytics.github.io/micro-sam/micro_sam.html), an AI-based napari plugin, can help with these more challenging images.

## Software setup

We will create a project folder, install the software and work in napari’s console and graphical user interface.

### 1. Open a terminal

- **Windows:** search for **PowerShell** in the Start menu and open it.
- **macOS:** use Spotlight to search for **Terminal**, or open **Applications > Utilities > Terminal**.
- **Linux:** open your usual terminal application.

The setup commands below go in this terminal, not in napari’s Python console. Lines starting with `#` are comments.

### 2. Install uv

`uv` is a Python package and environment manager. Follow the [uv installation guide](https://docs.astral.sh/uv/getting-started/installation/) for your operating system, then reopen your terminal and check that it is available:

```sh
uv --version
```

If you already installed uv using its standalone installer, you can update it with `uv self update`. If you installed it through a package manager, update it through that manager instead.

### 3. Create your project and install the packages

Run the code block for your operating system in a terminal. These commands create a new `napari-ai-tutorial` folder with a [virtual environment](https://docs.astral.sh/uv/pip/environments/) that keeps the tutorial's packages separate from other projects.

**Windows (PowerShell)**

```powershell
# Create a project folder
mkdir napari-ai-tutorial
cd napari-ai-tutorial

# Create and activate a Python environment
uv venv --python 3.12
.\.venv\Scripts\Activate.ps1

# Install the packages
uv pip install "micro-sam==1.8.10" "napari[pyqt6]>=0.6,<1" "scikit-image>=0.25,<0.27" napari-console
```

**macOS / Linux**

```bash
# Create a project folder
mkdir napari-ai-tutorial
cd napari-ai-tutorial

# Create and activate a Python environment
uv venv --python 3.12
source .venv/bin/activate

# Install the packages
uv pip install "micro-sam==1.8.10" "napari[pyqt6]>=0.6,<1" "scikit-image>=0.25,<0.27" napari-console
```

Installation may take a few minutes.

### 4. Launch napari

Once installation is complete, run the following command in the same terminal:

```bash
napari
```

napari should open in a separate window. Leave the terminal running while you work.

**Opening napari again**

You only need to install the packages once. The next time you want to use napari:

1. Open a terminal and navigate to your `napari-ai-tutorial` folder.
2. Activate the environment using the command for your operating system:
   - Windows: `.\.venv\Scripts\Activate.ps1`
   - macOS / Linux: `source .venv/bin/activate`
3. Run `napari`.

### 5. Open napari's console

In napari, choose **Window > Console**, or click the **Show/Hide IPython console** button beneath the layer list.

<style>
.jp-RenderedHTMLCommon img {
    width: 500px;
    max-width: 100%;
    height: auto;
    display: block;
    margin: 1rem auto;
}
</style>

## Example 1: Traditional segmentation of fluorescent nuclei

We will begin with a fluorescence microscopy example. In napari, select **File > Open Sample > napari builtins > Cells (3D + 2Ch)**.

![Fluorescence microscopy image](fig/fluorescencecells.png)

*Fluorescence microscopy sample image, Cells (3D + 2Ch).*

This sample contains membrane and nuclei channels. We will focus on segmenting the nuclei. 

Run the following code in napari's Python console.

The code imports the functions we need and retrieves the nuclei image from the viewer. It then applies a **Gaussian filter** to smooth the image and reduce noise. Next, Otsu's method automatically selects an intensity **threshold**: pixels brighter than this threshold become our nuclei mask.

```python
from skimage.filters import gaussian, threshold_otsu
from skimage.morphology import erosion, ball, disk
from skimage.measure import label
from skimage.segmentation import expand_labels

nuclei = viewer.layers["nuclei"].data
blurred_nuclei = gaussian(nuclei, sigma=3)
nuclei_mask = blurred_nuclei > threshold_otsu(blurred_nuclei)
viewer.add_image(blurred_nuclei, name="Smoothed nuclei")
viewer.add_labels(nuclei_mask, name="Nuclei mask")
```

![Fluorescence microscopy image](fig/fluorescencecells-smoothed_nuclei_and_mask.png)

*Smoothed nuclei channel and mask.*

After running the code, you should see the smoothed nuclei image and its mask in napari, as shown above.

Some nuclei are touching, so simply labelling connected regions would merge them into a single object. 

To separate them, we will **erode** the mask, shrinking the regions. Since our image is three-dimensional, we will use a ball-shaped footprint.

During the workshop, we found that a radius of 10 worked well for separating touching nuclei in this sample.

After separating the nuclei we will **label** the regions and **expand** them by the same distance.

```python
eroded_nuclei = erosion(nuclei_mask, footprint=ball(10))
nuclei_labels = label(eroded_nuclei)
nuclei_labels = expand_labels(nuclei_labels, distance=10)
viewer.add_labels(nuclei_labels, name="Nuclei instances")
```

After running the code above in the napari console, you should see a new **Nuclei instances** layer in napari, with each nucleus assigned a different label and colour.

Unlike the binary mask, which simply separates nuclei from the background, this is an **instance segmentation**: each nucleus is identified as a separate object.

![Segmented nuclei in 2D](fig/fluorescencecells-instances.png)

*Individual nuclei shown in different colours, with the original binary mask still visible underneath.*

Because our image is three-dimensional, we can also visualise the segmented nuclei in 3D. In the example below the binary mask layer has been hidden.

![Rotating 3D segmentation](fig/fluorescencecells-instances3D.gif)

*3D visualisation of the segmented nuclei.*

This approach works well because the fluorescent nuclei stand out clearly from the background. 

But what happens when the cells are not labelled?

## Example 2: Traditional segmentation of unlabelled live cells

Now let's try segmenting unlabelled live cells. We'll use an image from the [LIVECell dataset](https://doi.org/10.1038/s41592-021-01249-6).

Keep napari open so we can reuse the imports from Example 1, but remove the previous layers using the delete button in the layer list.

Next, go to **File > Open Sample > Segment Anything for Microscopy > LiveCell 2d example data**.

![Unlabelled live cells from the LIVECell dataset](fig/livecells.png)

*Unlabelled live cells from the LIVECell dataset.*

Take a moment to look at the image.

Unlike the fluorescent nuclei, these cells vary in appearance, and their boundaries are not always easy to distinguish from the background.

Run the code block below in the napari console to apply the same segmentation workflow as in Example 1: Gaussian smoothing, Otsu thresholding, erosion, labelling, and label expansion.

There are just two main differences. This time, we'll select pixels **darker** than the threshold using `<`, and because the image is two-dimensional, we'll use a disk-shaped footprint instead of a ball, with a smaller radius of 3 pixels instead of 10.

```python
live_image = viewer.layers["livecell"].data
blurred_live = gaussian(live_image, sigma=3)
live_mask = blurred_live < threshold_otsu(blurred_live)
eroded_live = erosion(live_mask, footprint=disk(3))
traditional_live = expand_labels(label(eroded_live), distance=3)
viewer.add_labels(traditional_live, name="Traditional segmentation")
```

![Traditional LIVECell segmentation with an over-segmented cell highlighted](fig/livecells-traditional-segmentation-mistakes.png)

*The red box highlights a cell split into several labels.*

While many cell regions have been identified, some cells have been largely missed or broken apart. This splitting is called **over-segmentation**.

In fluorescence imaging, traditional methods usually work well because labeled structures appear bright and distinct. But in label‑free imaging, cells are more heterogeneous in appearance and often have only subtle intensity differences from the background (with some areas being brighter, and others darker in this sample), making thresholding unreliable.

AI‑based segmentation methods can address these limitations.

## Example 3: AI-based segmentation with micro-SAM

AI-based segmentation takes a different approach. Instead of relying on fixed rules, such as intensity thresholds, models learn to recognise cells from training images paired with masks that outline individual cells. Once trained, they can segment cells in new images, even when the boundaries are faint or difficult to distinguish from the background.

How well a model performs on a new image depends on how similar that image is to the images it was trained on.

We will use [Segment Anything for Microscopy (micro-SAM, or μSAM)](https://computational-cell-analytics.github.io/micro-sam/micro_sam.html), which adapts Meta's Segment Anything Model for microscopy images. 

### Getting started with micro-SAM

Let's start with a fresh napari session to avoid interference from the layers we created in the previous examples.

Close napari and reopen it. 

Then load the LIVECell image again using **File > Open Sample > Segment Anything for Microscopy > LiveCell 2d example data**.

Next, open **Plugins > Segment Anything for Microscopy > Annotator 2d**. This opens the micro-SAM annotator and adds several layers to napari, including `prompts`, `point_prompts`, `current_object`, `committed_objects` and `auto_segmentation`.

Before we can start segmenting, micro-SAM needs to compute the image's **embeddings**. These are representations of the image that capture features the model uses to identify objects.

In the annotator:

1. Select the `livecell` image.
2. Under **Model**, choose **Light Microscopy**.
3. Click **Compute Embeddings** and wait for the process to finish.

Computing embeddings can take a little while, but they can be reused when we try different prompts on the same image.
### Segmenting a cell with a box prompt

Let's start by using a box prompt to segment a single cell.

1. Select the `prompts` layer in napari and choose the **rectangle tool** from the layer controls.
2. Draw a box around the cell you want to segment, as shown below.
3. In the micro-SAM annotator, click **Segment Object**.
4. Check the predicted mask in the `current_object` layer. If you're happy with the result, click **Commit** to add it to `committed_objects`.


![Box prompt around a cell](fig/livecells-box_prompt.png)

*Step 2: A box prompt drawn around the cell we want to segment.*

![Cell segmented using a box prompt](fig/livecells-box_prompt_segmented.png)

*Step 3 & 4: The resulting cell segmentation generated by micro-SAM using the box prompt.*

Try drawing boxes around a few more cells. You can commit each segmentation to build up a collection of individually labelled cells.

![Multiple cells segmented using box prompts](fig/livecells-box_prompt_segmented_multiple.png)

*Several cells segmented using individual box prompts.*

Notice that the boxes only tell micro-SAM **where to look**. The model predicts the actual cell boundaries, which do not have to follow the rectangular shape.

### Segmenting with point prompts

Instead of drawing boxes, we can guide micro-SAM using **positive and negative points**.

A positive point tells the model which object to include, while a negative point indicates a region to exclude. This can be useful when cells are touching or their boundaries are unclear.

To try it:

1. Clear any previous prompts and select the `point_prompts` layer.
2. Choose a **positive** prompt in the annotator and select the add-points tool in the layer controls.
3. Click inside a cell to place a point.
4. Click **Segment Object** to see the proposed mask.
5. If you're happy with the result, click **Commit**.

![Positive point prompt](fig/livecells-point_prompt.png)

*Step 3: A positive point placed inside the cell.*

![Cell segmented using a point prompt](fig/livecells-point_prompt_segmented.png)

*Step 4 & 5: Micro-SAM identifies the cell from a single positive point.*

If the predicted mask includes an unwanted region, you can add a **negative point** there and segment again to refine the result.

### Segmenting multiple cells with batched prompts

What happens if we place positive points inside several different cells?

By default, micro-SAM interprets all the points as belonging to **one object**. Try placing a point inside each of several cells and clicking **Segment Object**.

![Multiple point prompts without batching](fig/livecells-point_prompt_segmented_not_batched.png)

*Without batching, micro-SAM attempts to create a single mask that includes all the positive points.*

Now enable the **batched** option in the annotator and click **Segment Object** again.

![Multiple point prompts with batching](fig/livecells-point_prompt_segmented_batched.png)

*With batching enabled, micro-SAM predicts a separate mask for each positive point.*

This makes it much easier to segment several cells at once, without drawing individual boxes around them. Notice how the same points produce very different results depending on whether **batched** is enabled.

### Automatic segmentation

Finally, let's see what happens when we ask micro-SAM to segment the entire image without any prompts.

In the annotator, click **Automatic Segmentation**. Micro-SAM will predict masks for cells across the image and display them in the `auto_segmentation` layer.

Hide the other segmentation and annotation layers to see the result more clearly.

![Automatic micro-SAM segmentation of the LIVECell image](fig/livecells-automatic_segmentation.png)

*Automatic segmentation of the LIVECell image using micro-SAM.*

Micro-SAM does a better job of identifying individual cells in this image compared to the traditional methods used in Example 2.

## Final thoughts

Traditional image analysis methods worked well for segmenting fluorescent nuclei, but struggled with the unlabelled live cells.

Micro-SAM uses deep learning models to tackle these more challenging images, offering both interactive prompts and automatic segmentation. 

However, computing embeddings can take time, especially for larger images or models.

Traditional methods are often faster and simpler, so if they give good results, there's no need to use an AI-based approach.

As a rule of thumb, start simple, see how well it works and use more advanced tools when you need them. 

## Further resources

If you'd like to explore what has been covered in this tutorial in more detail or learn more about microscopy image analysis, here are some useful resources:

- [AI Catalyst workshop materials](https://healthbioscienceideas.github.io/napari-taster-aicatalyst/): the original workshop materials and exercises, if you'd like to revisit or explore the examples from this tutorial further.
- [Microscopy Image Analysis workshop](https://healthbioscienceideas.github.io/microscopy-novice/): our longer, two-day hands-on workshop covering image analysis with napari.
- [Introduction to Bioimage Analysis](https://bioimagebook.github.io/) by Pete Bankhead: an excellent free online book covering the principles and practical techniques of bioimage analysis.
- [micro-SAM documentation](https://computational-cell-analytics.github.io/micro-sam/micro_sam.html): guides to using micro-SAM, including additional features such as 3D annotation, tracking and model fine-tuning.
- [micro-SAM video tutorials](https://www.youtube.com/playlist?list=PLwYZXQJ3f36GQPpKCrSbHjGiH39X4XjSO): step-by-step demonstrations of the napari plugin and its segmentation tools.

## References

Archit, A., Freckmann, L., Nair, S., et al. (2025). Segment Anything for Microscopy. *Nature Methods*, **22**, 579–591. [https://doi.org/10.1038/s41592-024-02580-4](https://doi.org/10.1038/s41592-024-02580-4)

Edlund, C., Jackson, T. R., Khalid, N., et al. (2021). LIVECell—A large-scale dataset for label-free live cell segmentation. *Nature Methods*, **18**, 1038–1045. [https://doi.org/10.1038/s41592-021-01249-6](https://doi.org/10.1038/s41592-021-01249-6)